# Data Preparation

Clean, integrate, and aggregate the raw datasets to the Store-Week level for time-series analysis and forecasting.

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

In [3]:
def find_project_root():
    path = Path.cwd().resolve()

    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError("Project root not found")


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [4]:
train = pd.read_csv(RAW_DIR / "train.csv")
features = pd.read_csv(RAW_DIR / "features.csv")
stores = pd.read_csv(RAW_DIR / "stores.csv")

## Data Types

In [6]:
train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

In [7]:
assert pd.api.types.is_datetime64_any_dtype(train["Date"])
assert pd.api.types.is_datetime64_any_dtype(features["Date"])

assert train["Store"].dtype == "int64"
assert train["Dept"].dtype == "int64"

## Key Validation

In [9]:
feature_keys = features[["Store", "Date"]]

assert not feature_keys.duplicated().any()

print("Unique Store-Date feature records:", len(feature_keys))

Unique Store-Date feature records: 8190


In [10]:
assert not stores["Store"].duplicated().any()
assert stores["Store"].nunique() == train["Store"].nunique()

print("Stores validated:", stores["Store"].nunique())

Stores validated: 45


## Dataset Integration

In [12]:
data = train.merge(
    features,
    on=["Store", "Date"],
    how="left",
    suffixes=("", "_feature"),
    validate="many_to_one"
)

In [13]:
train_store_dates = train[["Store", "Date"]].drop_duplicates()

feature_matches = train_store_dates.merge(
    features[["Store", "Date"]],
    on=["Store", "Date"],
    how="left",
    indicator=True
)

assert (feature_matches["_merge"] == "both").all()

print("All training Store-Date records matched to features.")

All training Store-Date records matched to features.


In [14]:
assert (data["IsHoliday"] == data["IsHoliday_feature"]).all()

data = data.drop(columns="IsHoliday_feature")

In [15]:
data = data.merge(
    stores,
    on="Store",
    how="left",
    validate="many_to_one"
)

In [16]:
assert len(data) == len(train)
assert data["Type"].notna().all()
assert data["Size"].notna().all()

print("Merged rows:", len(data))

Merged rows: 421570


## Store-Week Aggregation

In [18]:
feature_columns = [
    "Temperature",
    "Fuel_Price",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5",
    "CPI",
    "Unemployment"
]

In [19]:
store_week = (
    data.groupby(["Store", "Date"], as_index=False)
    .agg(
        Weekly_Sales=("Weekly_Sales", "sum"),
        IsHoliday=("IsHoliday", "max"),
        **{
            col: (col, "first")
            for col in feature_columns
        },
        Type=("Type", "first"),
        Size=("Size", "first")
    )
)

In [20]:
assert not store_week.duplicated(["Store", "Date"]).any()

print("Store-Week rows:", len(store_week))
print("Stores:", store_week["Store"].nunique())
print("Weeks:", store_week["Date"].nunique())

Store-Week rows: 6435
Stores: 45
Weeks: 143


In [21]:
store_counts = store_week.groupby("Store")["Date"].nunique()

print("Minimum weeks per store:", store_counts.min())
print("Maximum weeks per store:", store_counts.max())

assert store_counts.nunique() == 1

Minimum weeks per store: 143
Maximum weeks per store: 143


## Sales Reconciliation

In [23]:
raw_sales = train["Weekly_Sales"].sum()
aggregated_sales = store_week["Weekly_Sales"].sum()

print("Raw sales:", raw_sales)
print("Store-Week sales:", aggregated_sales)

assert np.isclose(raw_sales, aggregated_sales)

Raw sales: 6737218987.110001
Store-Week sales: 6737218987.11


In [24]:
store_week = (
    store_week
    .sort_values(["Store", "Date"])
    .reset_index(drop=True)
)

store_week.head()

,Store,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,2010-02-05,1643690.90,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,2010-02-12,1641957.44,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,2010-02-19,1611968.17,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,2010-02-26,1409727.59,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,2010-03-05,1554806.68,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


## Missing Values After Integration

In [26]:
missing = (
    store_week.isna()
    .sum()
    .to_frame("Missing")
)

missing[missing["Missing"] > 0]

,Missing
MarkDown1,4155
MarkDown2,4798
MarkDown3,4389
MarkDown4,4470
MarkDown5,4140


## Save Prepared Dataset

In [28]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "store_week.csv"

store_week.to_csv(output_path, index=False)

print("Prepared dataset saved successfully.")

Prepared dataset saved successfully.


In [29]:
saved_data = pd.read_csv(output_path)

assert saved_data.shape == store_week.shape
assert saved_data.duplicated(["Store", "Date"]).sum() == 0

print("Final dataset:", saved_data.shape)

Final dataset: (6435, 15)


## Output

`data/processed/store_week.csv`

The dataset contains one observation per Store-Week and is ready for time-series analysis and feature engineering.